# 접시 전체 보존용 YOLO11-seg 학습

이 노트북은 AIHub 정제 이미지에서 `plate_full`과 `food_visible` 마스크를 학습합니다. CVAT 주석은 사람이 수행해야 합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
SOURCE_ROOT = Path('/content/drive/MyDrive/final_1_team/data/processed/aihub_food_image_text/v2/food_description_data')
assert PROJECT_ROOT.is_dir(), PROJECT_ROOT
assert (SOURCE_ROOT / 'metadata.csv').is_file(), SOURCE_ROOT
%cd {PROJECT_ROOT}

In [ ]:
!pip install --prefer-binary -r requirements-colab.txt
!pip install pycocotools
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')

In [ ]:
# 최초 1회 실행: AIHub 원본에서 주석 후보 500장을 준비합니다.
!python -m scripts.prepare_plate_annotation_manifest --source-root {SOURCE_ROOT} --sample-size 500
# 생성된 data/training/plate_segmentation/cvat_images 를 CVAT에 올려
# plate_full, food_visible을 주석한 후 COCO Instances 1.0 JSON을
# data/training/plate_segmentation/annotations/instances_default.json 에 저장합니다.

In [ ]:
# CSV의 target_split(train/val/test)과 상태(completed/skipped)를 모두 작성한 다음 실행합니다.
!python -m scripts.prepare_plate_segmentation_dataset --coco-json data/training/plate_segmentation/annotations/instances_default.json --images-dir data/training/plate_segmentation/cvat_images --copy-images
!python -m scripts.train_yolo11n_plate_segmenter --epochs 100 --imgsz 1024
!python -m scripts.evaluate_yolo11n_plate_segmenter --weights runs/plate_segmenter/yolo11n_plate_seg_v1/weights/best.pt

In [ ]:
# 검수 후 가중치를 운영 경로로 복사하고 pipeline.yaml의 plate_segmenter.enabled를 true로 설정합니다.
!cp runs/plate_segmenter/yolo11n_plate_seg_v1/weights/best.pt models/yolo11n_plate_seg.pt